# 🧠 AGSE-VNet — Brain Atlas Mapping
## Epoch-55 Model → Julich Brain Atlas (Paper-Exact Implementation)

Implements **Section 3.4** of *"From segmentation to explanation: Generating textual reports from MRI with LLMs"* (Valerio et al., 2025) using our AGSE-VNet segmentation model.

**Pipeline:**
1. Load epoch-55 best checkpoint → run inference on a patient  
2. Register native-space segmentation → MNI152 standard space (ANTsPy)  
3. Map Tumor Core voxels → Julich Brain Atlas regions (siibra-python)  
4. Compute: % of tumor per region + % of each region affected  
5. Rank top-5 most affected regions  
6. Export structured JSON (paper Listing 1 format)  

---
| Label | Sub-region | Paper Convention |
|---|---|---|
| 1 | NCR – Necrotic Core | Tumor Core (TC) |
| 2 | ED – Peritumoral Edema | Peritumoral Edema |
| 3 | ET – Enhancing Tumor | GD-Enhancing Tumor |

> **TC (Tumor Core) = labels 1 + 3** — exactly as defined in BraTS/paper.


In [ ]:
# Cell 1 — Install Dependencies
import subprocess, sys

pkgs = [
    'nibabel',
    'monai',
    'einops',
    'scikit-learn',
    'tqdm',
    'antspyx',        # MNI152 registration
    'nilearn',        # MNI152 template download + resampling
    'siibra',         # Julich Brain Atlas (paper-exact library)
]
for pkg in pkgs:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('✅ All packages installed.')


In [ ]:
# Cell 2 — Imports & Reproducibility
import os, gc, json, random, time, warnings
warnings.filterwarnings('ignore')

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast

import ants
import siibra
from nilearn import datasets, image as nlimage

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


In [ ]:
# Cell 3 — Mount Drive & Paths
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT     = Path('/content/drive/MyDrive')
DATASET_DIR    = DRIVE_ROOT / 'BraTS2021_Dataset'
CHECKPOINT_DIR = DRIVE_ROOT / 'AGSE_UNet3D_Models'
OUTPUT_DIR     = DRIVE_ROOT / 'AGSE_AtlasMapping_Results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CKPT_BEST = CHECKPOINT_DIR / 'best_model.pth'
assert CKPT_BEST.exists(), f'Checkpoint not found: {CKPT_BEST}'
print(f'Checkpoint : {CKPT_BEST}')
print(f'Dataset    : {DATASET_DIR}')
print(f'Output     : {OUTPUT_DIR}')


In [ ]:
# Cell 4 — Config (must exactly match training)
CFG = dict(
    modalities      = ['flair', 't1ce', 't2', 't1'],
    num_classes     = 4,
    patch_size      = (96, 96, 96),
    base_filters    = 16,
    se_reduction    = 4,
    dropout         = 0.1,
    mixed_precision = True,
)

# ── BraTS label conventions (paper Section 3.2) ──
# Our model labels:  0=BG  1=NCR  2=Edema  3=ET
# Paper sub-regions:
#   TC (Tumor Core)    = NCR + ET = labels {1, 3}
#   WT (Whole Tumor)   = NCR + Edema + ET = labels {1, 2, 3}
#   ET (Enhancing)     = label {3}

LABEL = dict(
    BACKGROUND = 0,
    NCR        = 1,   # Necrotic Core  → part of Tumor Core
    EDEMA      = 2,   # Peritumoral Edema
    ET         = 3,   # GD-Enhancing Tumor → part of Tumor Core
)

TC_LABELS = {LABEL['NCR'], LABEL['ET']}    # Tumor Core
WT_LABELS = {LABEL['NCR'], LABEL['EDEMA'], LABEL['ET']}  # Whole Tumor
ET_LABELS = {LABEL['ET']}                  # Enhancing Tumor

# Color coding as defined in paper Listing 1
COLOR_MAP = {
    'Tumor_Core'       : 'red',
    'Peritumoral_Edema': 'yellow',
    'GD_Enhancing_Tumor': 'green',
}
print('Config loaded.')


In [ ]:
# Cell 5 — 3D UNet-AGSE Architecture (identical to training)

class ConvBnReLU(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel, stride, padding, bias=False),
            nn.BatchNorm3d(out_ch), nn.ReLU(inplace=True))
    def forward(self, x): return self.block(x)

class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False), nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False), nn.Sigmoid())
    def forward(self, x):
        B, C = x.shape[:2]
        return x * self.fc(x.view(B,C,-1).mean(-1)).view(B,C,1,1,1)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, se_reduction=4, dropout=0.1):
        super().__init__()
        self.conv1    = ConvBnReLU(in_ch, out_ch)
        self.conv2    = nn.Sequential(
            nn.Conv3d(out_ch, out_ch, 3, 1, 1, bias=False), nn.BatchNorm3d(out_ch))
        self.se       = SEBlock3D(out_ch, se_reduction)
        self.drop     = nn.Dropout3d(dropout)
        self.relu     = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv3d(in_ch,out_ch,1,bias=False),
                          nn.BatchNorm3d(out_ch)) if in_ch!=out_ch else nn.Identity())
    def forward(self, x):
        return self.relu(self.se(self.drop(self.conv2(self.conv1(x)))) + self.shortcut(x))

class AttentionGate3D(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv3d(F_g,F_int,1), nn.BatchNorm3d(F_int))
        self.W_x  = nn.Sequential(nn.Conv3d(F_l,F_int,1), nn.BatchNorm3d(F_int))
        self.psi  = nn.Sequential(nn.Conv3d(F_int,1,1), nn.BatchNorm3d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape != x1.shape:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='trilinear', align_corners=False)
        return x * self.psi(self.relu(g1 + x1))

class UNet3D_AGSE(nn.Module):
    def __init__(self, in_ch=4, num_classes=4, base_filters=16, se_r=4, dropout=0.1):
        super().__init__()
        F = base_filters
        self.enc1=ResidualBlock(in_ch,F,se_r,dropout)
        self.enc2=ResidualBlock(F,F*2,se_r,dropout)
        self.enc3=ResidualBlock(F*2,F*4,se_r,dropout)
        self.enc4=ResidualBlock(F*4,F*8,se_r,dropout)
        self.down1=nn.Conv3d(F,F,2,stride=2,bias=False)
        self.down2=nn.Conv3d(F*2,F*2,2,stride=2,bias=False)
        self.down3=nn.Conv3d(F*4,F*4,2,stride=2,bias=False)
        self.down4=nn.Conv3d(F*8,F*8,2,stride=2,bias=False)
        self.bottleneck=ResidualBlock(F*8,F*16,se_r,dropout)
        self.ag4=AttentionGate3D(F*8,F*8,F*4)
        self.ag3=AttentionGate3D(F*4,F*4,F*2)
        self.ag2=AttentionGate3D(F*2,F*2,F)
        self.ag1=AttentionGate3D(F,F,F//2)
        self.up4=nn.ConvTranspose3d(F*16,F*8,2,stride=2)
        self.dec4=ResidualBlock(F*16,F*8,se_r,dropout)
        self.up3=nn.ConvTranspose3d(F*8,F*4,2,stride=2)
        self.dec3=ResidualBlock(F*8,F*4,se_r,dropout)
        self.up2=nn.ConvTranspose3d(F*4,F*2,2,stride=2)
        self.dec2=ResidualBlock(F*4,F*2,se_r,dropout)
        self.up1=nn.ConvTranspose3d(F*2,F,2,stride=2)
        self.dec1=ResidualBlock(F*2,F,se_r,dropout)
        self.out=nn.Conv3d(F,num_classes,1)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m,nn.Conv3d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu')
            elif isinstance(m,nn.BatchNorm3d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.down1(e1))
        e3=self.enc3(self.down2(e2)); e4=self.enc4(self.down3(e3))
        b=self.bottleneck(self.down4(e4))
        d4=self.dec4(torch.cat([self.up4(b),  self.ag4(self.up4(b),e4)],1))
        d3=self.dec3(torch.cat([self.up3(d4), self.ag3(self.up3(d4),e3)],1))
        d2=self.dec2(torch.cat([self.up2(d3), self.ag2(self.up2(d3),e2)],1))
        d1=self.dec1(torch.cat([self.up1(d2), self.ag1(self.up1(d2),e1)],1))
        return self.out(d1)

model = UNet3D_AGSE(
    in_ch=len(CFG['modalities']), num_classes=CFG['num_classes'],
    base_filters=CFG['base_filters'], se_r=CFG['se_reduction'], dropout=CFG['dropout']
).to(DEVICE)
print(f'Model built — {sum(p.numel() for p in model.parameters()):,} parameters')


In [ ]:
# Cell 6 — Load Epoch-55 Best Model Checkpoint
ckpt = torch.load(CKPT_BEST, map_location=DEVICE)

# Handle different checkpoint formats
if 'model' in ckpt:
    model.load_state_dict(ckpt['model'])
elif 'model_state_dict' in ckpt:
    model.load_state_dict(ckpt['model_state_dict'])
elif 'state_dict' in ckpt:
    model.load_state_dict(ckpt['state_dict'])  # ← Your checkpoint
else:
    model.load_state_dict(ckpt)

model.eval()

saved_epoch  = ckpt.get('epoch', 54)
best_dice_wt = ckpt.get('best', 0.0)
history      = ckpt.get('history', {})

print('=' * 52)
print(f'  Checkpoint : {CKPT_BEST.name}')
print(f'  Epoch      : {int(saved_epoch)+1}')
print(f'  Best Dice-WT: {best_dice_wt:.4f}')
# ... rest of your original code

In [ ]:
# Cell 7 — Select Patient & Run Sliding-Window Inference

def znorm(vol):
    mask = vol > 0
    if not mask.any(): return vol
    vol = (vol - vol[mask].mean()) / (vol[mask].std() + 1e-8)
    vol *= mask
    return vol


def sliding_window_inference(model, volume, patch=(96,96,96), stride=48, device=DEVICE):
    """Gaussian-weighted sliding window — BraTS evaluation protocol."""
    _, D, H, W = volume.shape
    pd, ph, pw = patch

    # Gaussian weight kernel
    def g1d(n): x=np.linspace(-1,1,n); return np.exp(-2*x**2)
    gauss = torch.tensor(
        np.einsum('i,j,k->ijk', g1d(pd), g1d(ph), g1d(pw)).astype(np.float32)
    ).to(device)

    acc   = torch.zeros(CFG['num_classes'], D, H, W, device=device)
    count = torch.zeros(D, H, W, device=device)

    model.eval()
    with torch.no_grad():
        for d0 in range(0, max(D-pd+1,1), stride):
            d0 = min(d0, max(D-pd,0))
            for h0 in range(0, max(H-ph+1,1), stride):
                h0 = min(h0, max(H-ph,0))
                for w0 in range(0, max(W-pw+1,1), stride):
                    w0 = min(w0, max(W-pw,0))
                    pat = volume[:, d0:d0+pd, h0:h0+ph, w0:w0+pw]
                    dp,hp,wp = pat.shape[1:]
                    if dp<pd or hp<ph or wp<pw:
                        pat = np.pad(pat,[(0,0),(0,pd-dp),(0,ph-hp),(0,pw-wp)])
                    t = torch.tensor(pat[None], dtype=torch.float32).to(device)
                    with autocast(enabled=CFG['mixed_precision'] and device.type=='cuda'):
                        prob = torch.softmax(model(t)[0], dim=0)
                    acc[:, d0:d0+pd, h0:h0+ph, w0:w0+pw] += prob * gauss
                    count[   d0:d0+pd, h0:h0+ph, w0:w0+pw] += gauss

    return (acc / count.unsqueeze(0).clamp(1e-5)).cpu().numpy()


# ── Pick patient (change index to select a different case) ─────────────────
PATIENT_IDX = 45   # ← adjust as needed
all_pts      = sorted([d for d in DATASET_DIR.iterdir() if d.is_dir()])
PAT_DIR      = all_pts[PATIENT_IDX]
PAT_NAME     = PAT_DIR.name
print(f'Patient: {PAT_NAME}  ({PATIENT_IDX+1}/{len(all_pts)})')

# Load raw NIfTI objects (keep affine & header for later export)
raw_nibs = {m: nib.load(str(PAT_DIR / f'{PAT_NAME}_{m}.nii.gz'))
            for m in CFG['modalities']}
NATIVE_AFFINE = raw_nibs['flair'].affine
NATIVE_HEADER = raw_nibs['flair'].header
SPACING       = np.sqrt((NATIVE_AFFINE[:3,:3]**2).sum(0))   # mm per voxel
VOX_VOL_MM3   = float(np.prod(SPACING))

# Build normalised 4-channel volume
volume = np.stack([znorm(raw_nibs[m].get_fdata(dtype=np.float32))
                   for m in CFG['modalities']], axis=0)   # (4,D,H,W)

# Ground-truth segmentation
seg_gt = nib.load(str(PAT_DIR / f'{PAT_NAME}_seg.nii.gz')).get_fdata().astype(np.int32)
seg_gt[seg_gt == 4] = 3    # BraTS label remap: 4 → 3

print(f'Volume shape : {volume.shape}')
print(f'GT seg unique: {np.unique(seg_gt)}')
print(f'Voxel spacing: {SPACING} mm')

print('\nRunning sliding-window inference...')
t0 = time.time()
prob_map = sliding_window_inference(model, volume)
seg_pred = prob_map.argmax(0).astype(np.int32)   # (D,H,W)
print(f'Done in {time.time()-t0:.0f}s  |  unique labels: {np.unique(seg_pred)}')

# Quick Dice check
def dice(pred, gt, lbl_set):
    p = np.isin(pred, list(lbl_set)).astype(float)
    g = np.isin(gt,   list(lbl_set)).astype(float)
    return 2*(p*g).sum() / (p.sum()+g.sum()+1e-5)

RUN_DICE = {
    'ET': dice(seg_pred, seg_gt, ET_LABELS),
    'TC': dice(seg_pred, seg_gt, TC_LABELS),
    'WT': dice(seg_pred, seg_gt, WT_LABELS),
}
RUN_DICE['Average'] = float(np.mean(list(RUN_DICE.values())))
print('\nInference Dice (this patient):')
for k,v in RUN_DICE.items():
    print(f'  {k}: {v:.4f}')


In [ ]:
# Cell 8 — Save Native-Space Segmentation + FLAIR for Registration Input

# Full predicted segmentation (all labels)
seg_pred_nii = nib.Nifti1Image(seg_pred.astype(np.int16), NATIVE_AFFINE, NATIVE_HEADER)
nib.save(seg_pred_nii, '/tmp/seg_pred_native.nii.gz')

# Tumor Core mask (labels 1+3) — used for atlas mapping (paper Section 3.4)
tc_mask_native = np.isin(seg_pred, list(TC_LABELS)).astype(np.int16)
tc_nii = nib.Nifti1Image(tc_mask_native, NATIVE_AFFINE, NATIVE_HEADER)
nib.save(tc_nii, '/tmp/tc_mask_native.nii.gz')

# Peritumoral Edema mask
ed_mask_native = (seg_pred == LABEL['EDEMA']).astype(np.int16)
nib.save(nib.Nifti1Image(ed_mask_native, NATIVE_AFFINE, NATIVE_HEADER),
         '/tmp/ed_mask_native.nii.gz')

# ET mask
et_mask_native = (seg_pred == LABEL['ET']).astype(np.int16)
nib.save(nib.Nifti1Image(et_mask_native, NATIVE_AFFINE, NATIVE_HEADER),
         '/tmp/et_mask_native.nii.gz')

# Patient FLAIR (for registration as the moving image)
flair_path = str(PAT_DIR / f'{PAT_NAME}_flair.nii.gz')

print('Native-space masks saved:')
print(f'  Tumor Core voxels : {int(tc_mask_native.sum())}')
print(f'  Edema voxels      : {int(ed_mask_native.sum())}')
print(f'  ET voxels         : {int(et_mask_native.sum())}')


In [ ]:
# Cell 9 — Register Patient FLAIR → MNI152 Space (ANTsPy)
#
# Paper Section 3.4:
#   "The mapping process involved overlaying the ROIs onto the
#    standard MNI152 brain space."
#
# We perform SyNQuick (affine + fast diffeomorphic) registration.
# The SAME transforms are then applied to all three tumor masks.

print('Step 1/3 — Fetching MNI152 T1 template (nilearn)...')
mni_dataset = datasets.fetch_icbm152_2009()
MNI_PATH    = mni_dataset.t1
print(f'  MNI152 template: {MNI_PATH}')

print('Step 2/3 — Loading images into ANTsPy...')
fixed_img  = ants.image_read(MNI_PATH)         # MNI152 (target / fixed)
moving_img = ants.image_read(flair_path)        # Patient FLAIR (source / moving)
print(f'  Fixed  (MNI152): shape={fixed_img.shape}  spacing={fixed_img.spacing}')
print(f'  Moving (patient): shape={moving_img.shape}  spacing={moving_img.spacing}')

print('Step 3/3 — Running ANTsPy SyN registration (~3-8 min)...')
t_reg = time.time()
reg = ants.registration(
    fixed             = fixed_img,
    moving            = moving_img,
    type_of_transform = 'SyN',
    grad_step         = 0.1,
    flow_sigma        = 3,
    total_sigma       = 0,
    verbose           = False,
)
FWD_TRANSFORMS = reg['fwdtransforms']   # forward transform list for apply_transforms
print(f'Registration complete in {time.time()-t_reg:.0f}s')

# ── Warp all masks to MNI152 using nearest-neighbour (preserve labels) ─────
def warp_mask(native_path, transforms, fixed, out_path):
    moving   = ants.image_read(native_path)
    warped   = ants.apply_transforms(
        fixed        = fixed,
        moving       = moving,
        transformlist= transforms,
        interpolator = 'nearestNeighbor',
    )
    ants.image_write(warped, out_path)
    return warped.numpy().astype(np.int32)

print('\nWarping masks to MNI152 space...')
tc_mni = warp_mask('/tmp/tc_mask_native.nii.gz', FWD_TRANSFORMS, fixed_img,
                    '/content/tc_mask_mni.nii.gz')
ed_mni = warp_mask('/tmp/ed_mask_native.nii.gz', FWD_TRANSFORMS, fixed_img,
                    '/content/ed_mask_mni.nii.gz')
et_mni = warp_mask('/tmp/et_mask_native.nii.gz', FWD_TRANSFORMS, fixed_img,
                    '/content/et_mask_mni.nii.gz')

# Full seg in MNI (for visualisation)
seg_mni = warp_mask('/tmp/seg_pred_native.nii.gz', FWD_TRANSFORMS, fixed_img,
                     '/content/seg_pred_mni.nii.gz')
warped_flair = reg['warpedmovout']
ants.image_write(warped_flair, '/content/flair_mni.nii.gz')

print(f'  TC voxels in MNI  : {int(tc_mni.sum())}')
print(f'  Edema voxels in MNI: {int(ed_mni.sum())}')
print(f'  ET voxels in MNI  : {int(et_mni.sum())}')
print('All MNI-space masks saved to /content/')

In [ ]:
# Cell 10 — Julich Brain Atlas Mapping (siibra-python)
#
# Paper Section 3.4 exact procedure:
#   "we generated a point set representation of the tumor core voxels,
#    which were then assigned to the corresponding brain regions
#    in the labeled atlas map."
#
# siibra-python accesses the Julich Brain Atlas — the SAME library
# cited in the paper (ref [35]: Dickscheid et al., siibra-python, 2024).

print('Initialising Julich Brain Atlas via siibra...')
atlas        = siibra.atlases['human']
parcellation = atlas.get_parcellation('julich brain')
space_mni    = atlas.get_space('mni152')

# Fetch the labelled parcellation volume in MNI152 space
print('Fetching Julich Brain Atlas labelled volume in MNI152 space...')
pmap         = parcellation.get_map(space=space_mni, maptype='labelled')
atlas_vol_nib = pmap.fetch()                          # nibabel NIfTI image
atlas_data    = atlas_vol_nib.get_fdata(dtype=np.float32).astype(np.int32)
atlas_affine  = atlas_vol_nib.affine
print(f'  Atlas volume shape : {atlas_data.shape}')
print(f'  Unique region IDs  : {len(np.unique(atlas_data[atlas_data>0]))}')

# ── Build label_id → region_name mapping ──────────────────────────────
print('Building ID → region name lookup from parcellation map...')
label_to_name = {}

if hasattr(pmap, 'regions') and pmap.regions:
    if isinstance(pmap.regions, dict):
        # Older siibra: dict of {label_idx: region}
        for label_idx, region in pmap.regions.items():
            label_to_name[int(label_idx)] = region.name
    else:
        # Newer siibra: list of region objects
        for region in pmap.regions:
            idx = getattr(region, 'index', None)
            if idx is not None:
                lbl = getattr(idx, 'label', None)
                if lbl is not None:
                    label_to_name[int(lbl)] = region.name

# Fallback: build from the atlas volume itself using the fetched NIfTI header
# The labelled volume has 207 unique IDs — map them by position if siibra gives no names
if not label_to_name:
    print('  siibra region names unavailable — using generic fallback labels')
    unique_ids = np.unique(atlas_data[atlas_data > 0])
    for uid in unique_ids:
        label_to_name[int(uid)] = f'Julich_Region_{int(uid)}'

print(f'  Region names loaded: {len(label_to_name)}')

In [ ]:
# Cell 11 — Resample TC Mask → Atlas Space & Assign Voxels to Julich Regions
#
# We must bring the TC mask and the atlas to the same voxel grid.
# Strategy: resample the atlas to our MNI-warped TC mask space
# (or vice versa — we resample the TC mask to the atlas grid, which
# is already in MNI152 space, making this a simple resampling).

from nilearn.image import resample_img

tc_nib_mni  = nib.load('/content/tc_mask_mni.nii.gz')

print('Resampling TC mask to Julich atlas grid...')
tc_resampled = resample_img(
    tc_nib_mni,
    target_affine = atlas_affine,
    target_shape  = atlas_data.shape,
    interpolation = 'nearest',
)
tc_in_atlas_space = tc_resampled.get_fdata(dtype=np.float32).astype(np.int32)
print(f'  TC mask resampled shape : {tc_in_atlas_space.shape}')
print(f'  TC voxels after resample: {int(tc_in_atlas_space.sum())}')

# ── Point-set representation of Tumor Core voxels ─────────────────────────
# "we generated a point set representation of the tumor core voxels,
#  which were then assigned to the corresponding brain regions" — paper
tc_voxel_coords = np.argwhere(tc_in_atlas_space > 0)   # (N, 3) voxel indices
print(f'  TC voxel coordinates    : {len(tc_voxel_coords)} points')

# ── Assign each TC voxel to its Julich Brain Atlas region ─────────────────
TC_TOTAL_VOXELS = len(tc_voxel_coords)
assert TC_TOTAL_VOXELS > 0, 'No Tumor Core voxels found — check segmentation or registration.'

region_voxel_count = {}    # {atlas_label_id: voxel_count}
atlas_flat = atlas_data.ravel()

for coord in tc_voxel_coords:
    atlas_id = int(atlas_data[coord[0], coord[1], coord[2]])
    if atlas_id == 0:   # background / unlabeled in atlas → skip
        continue
    region_voxel_count[atlas_id] = region_voxel_count.get(atlas_id, 0) + 1

print(f'  TC voxels with atlas label: {sum(region_voxel_count.values())}')
print(f'  Unique Julich regions hit : {len(region_voxel_count)}')


In [ ]:
# Cell 12 — Compute Percentages & Extract Top-5 Regions
#
# Two metrics per region (paper exact definition):
#   Percentage_of_Tumor    = (TC voxels in region / total TC voxels) × 100
#   Percentage_of_Region_Affected = (TC voxels in region / total atlas region size) × 100

# Pre-compute size of every atlas region (in the atlas-grid space)
print('Computing atlas region sizes...')
region_sizes = {}    # {atlas_id: total_voxels_in_that_region}
unique_ids, counts = np.unique(atlas_data[atlas_data > 0], return_counts=True)
for aid, cnt in zip(unique_ids, counts):
    region_sizes[int(aid)] = int(cnt)

# Build per-region records
records = []
for atlas_id, tc_count in region_voxel_count.items():
    region_name  = label_to_name.get(atlas_id, f'Region_{atlas_id}')
    region_size  = region_sizes.get(atlas_id, 1)

    pct_tumor    = 100.0 * tc_count / TC_TOTAL_VOXELS       # % of TC in this region
    pct_region   = 100.0 * tc_count / region_size            # % of this region affected

    records.append({
        'atlas_id'    : atlas_id,
        'Region'      : region_name,
        'tc_voxels'   : tc_count,
        'region_size' : region_size,
        'Percentage_of_Tumor'           : round(pct_tumor,  2),
        'Percentage_of_Region_Affected' : round(pct_region, 2),
    })

# Sort by % of tumor (largest first) — matches paper ordering
records.sort(key=lambda r: r['Percentage_of_Tumor'], reverse=True)

# Top-5 (paper uses exactly 5 regions)
TOP_N   = 5
top5    = records[:TOP_N]

print('\n' + '='*78)
print('  TOP-5 JULICH BRAIN ATLAS REGIONS  |  TUMOR CORE (NCR + ET)  |  AGSE-VNet Ep55')
print('='*78)
print(f'  {"#":<3} {"Region":<42} {"% Tumor":>9}  {"% Region Affected":>18}')
print('-'*78)
for i, r in enumerate(top5, 1):
    print(f'  {i:<3} {r["Region"]:<42} {r["Percentage_of_Tumor"]:>8.2f}%  {r["Percentage_of_Region_Affected"]:>17.2f}%')
print('='*78)


In [ ]:
# Cell 13 — Build Structured JSON (Paper Listing 1 exact format)
#
# The JSON serves as "source of truth" for subsequent LLM prompting
# (Section 3.5 — JSON construction).

spatial_distribution = [
    {
        'Region'                        : r['Region'],
        'tc_voxels'                     : r['tc_voxels'],  # Added for dashboard
        'Percentage_of_Tumor'           : f'{r["Percentage_of_Tumor"]:.2f}%',
        'Percentage_of_Region_Affected' : f'{r["Percentage_of_Region_Affected"]:.2f}%',
    }
    for r in top5
]

atlas_json = {
    'MRI_Scan': {
        'Patient'       : PAT_NAME,
        'Epoch_Loaded'  : int(saved_epoch) + 1,
        'Tumor_Details' : {
            'Spatial_Distribution': spatial_distribution,
            'Total_Tumor_Voxels': TC_TOTAL_VOXELS, # Added for dashboard

            # Color coding — paper Listing 1
            'Semantic_Segmentation': {
                'Tumor_Core'        : {'Color': COLOR_MAP['Tumor_Core'],
                                       'Labels': 'NCR (1) + ET (3)'},
                'Peritumoral_Edema' : {'Color': COLOR_MAP['Peritumoral_Edema'],
                                       'Labels': 'Edema (2)'},
                'GD_Enhancing_Tumor': {'Color': COLOR_MAP['GD_Enhancing_Tumor'],
                                       'Labels': 'ET (3)'},
            },

            # Segmentation confidence (paper Listing 1 fields)
            'Segmentation_Confidence': {
                'Dice_Score': {
                    'Enhancing_Tumor' : round(RUN_DICE['ET'],   4),
                    'Tumor_Core'      : round(RUN_DICE['TC'],   4),
                    'Whole_Tumor'     : round(RUN_DICE['WT'],   4),
                    'Average'         : round(RUN_DICE['Average'], 4),
                },
                'Note': 'Dice scores computed on this patient against ground truth',
            },

            'Model_Used': 'AGSE-VNet (3D UNet + Attention Gates + SE Blocks)',
            'Atlas_Used': 'Julich Brain Atlas (cytoarchitectonic probabilistic, MNI152)',
            'Atlas_Library': 'siibra-python',
        }
    }
}

json_str  = json.dumps(atlas_json, indent=2)
json_path = '/content/atlas_mapping_output.json'
with open(json_path, 'w') as f:
    f.write(json_str)

print(json_str)
print(f'\nJSON saved → {json_path}')

In [ ]:
# Cell 14 — Visualisation
#   (a) MNI-space FLAIR overlaid with TC mask (3 anatomical planes)
#   (b) Horizontal bar chart — top-5 Julich regions (paper Fig. 3 style)

flair_mni_nib = nib.load('/content/flair_mni.nii.gz')
tc_mni_nib    = nib.load('/content/tc_mask_mni.nii.gz')
seg_mni_nib   = nib.load('/content/seg_pred_mni.nii.gz')

# ✅ Fix — load as float32, then cast seg to int after
flair_d = flair_mni_nib.get_fdata(dtype=np.float32)
tc_d    = tc_mni_nib.get_fdata(dtype=np.float32)
seg_d   = seg_mni_nib.get_fdata(dtype=np.float32).astype(np.int32)

# Best axial, coronal, sagittal slices for TC
if tc_d.sum() > 0:
    tc_coords = np.argwhere(tc_d > 0)
    centre    = tc_coords.mean(0).astype(int)
else:
    centre = np.array(flair_d.shape) // 2

fig = plt.figure(figsize=(20, 10))
fig.patch.set_facecolor('#0d0d0d')
fig.suptitle(f'AGSE-VNet Epoch-55 | Julich Brain Atlas Mapping | {PAT_NAME}',
             fontsize=13, color='white', fontweight='bold', y=0.98)

LABEL_CMAP = {1: [1,.2,.2], 2: [1,1,0], 3: [.2,1,.2]}

def seg_rgb(slice_2d):
    h, w = slice_2d.shape
    rgb   = np.zeros((h, w, 4), dtype=np.float32)
    for lbl, col in LABEL_CMAP.items():
        m = slice_2d == lbl
        rgb[m, :3] = col; rgb[m, 3] = 0.55
    return rgb

ax_titles = ['Axial (TC centroid)', 'Coronal (TC centroid)', 'Sagittal (TC centroid)']
slices    = [
    (flair_d[centre[0],:,:],  seg_d[centre[0],:,:]),
    (flair_d[:,centre[1],:],  seg_d[:,centre[1],:]),
    (flair_d[:,:,centre[2]], seg_d[:,:,centre[2]]),
]

for col, (fslice, sslice), title in zip(range(3), slices, ax_titles):
    ax = fig.add_subplot(2, 3, col+1)
    ax.set_facecolor('black')
    ax.imshow(np.rot90(fslice), cmap='gray', aspect='auto')
    ax.imshow(np.rot90(seg_rgb(sslice)), aspect='auto')
    ax.set_title(title, color='white', fontsize=10)
    ax.axis('off')

legend_handles = [
    mpatches.Patch(color=[1,.2,.2], label='NCR / Tumor Core'),
    mpatches.Patch(color=[1,1,0],   label='Peritumoral Edema'),
    mpatches.Patch(color=[.2,1,.2], label='GD-Enhancing Tumor'),
]
fig.legend(handles=legend_handles, loc='upper center', ncol=3,
           bbox_to_anchor=(0.5, 0.92), fontsize=9, facecolor='#1a1a1a',
           labelcolor='white', edgecolor='gray')

# ── Bar chart (paper Fig. 3 style) ────────────────────────────────────────
ax_bar = fig.add_subplot(2, 3, (4, 6))   # spans bottom row
ax_bar.set_facecolor('#1a1a1a')

regions_short = [r['Region'][:45] for r in top5]
pct_tumor     = [r['Percentage_of_Tumor']           for r in top5]
pct_region    = [r['Percentage_of_Region_Affected']  for r in top5]

x = np.arange(len(top5))
w = 0.38
b1 = ax_bar.bar(x - w/2, pct_tumor,  w, label='% of Tumor in Region', color='#e05252', alpha=0.9, edgecolor='white', lw=0.5)
b2 = ax_bar.bar(x + w/2, pct_region, w, label='% of Region Affected',  color='#52abe0', alpha=0.9, edgecolor='white', lw=0.5)

for bar, val in zip(b1, pct_tumor):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8, color='white')
for bar, val in zip(b2, pct_region):
    ax_bar.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=8, color='white')

ax_bar.set_xticks(x)
ax_bar.set_xticklabels([f'#{i+1} {r[:30]}' for i,r in enumerate(regions_short)],
                        rotation=18, ha='right', fontsize=8, color='white')
ax_bar.set_ylabel('Percentage (%)', color='white')
ax_bar.set_title('Top-5 Julich Brain Atlas Regions — Tumor Core Distribution',
                  color='white', fontsize=11, fontweight='bold')
ax_bar.tick_params(colors='white'); ax_bar.spines[:].set_color('#555')
ax_bar.legend(fontsize=9, facecolor='#1a1a1a', labelcolor='white', loc='upper right')
ax_bar.set_facecolor('#1a1a1a')
ax_bar.yaxis.label.set_color('white')

plt.tight_layout(rect=[0,0,1,0.90])
plt.savefig('/content/atlas_mapping_visualization.png', dpi=150,
            bbox_inches='tight', facecolor='#0d0d0d')
plt.show()
print('Saved: /content/atlas_mapping_visualization.png')


In [ ]:
# Cell 15 — Print Summary Report & Export Everything to Drive

print('\n' + '='*70)
print('  BRAIN ATLAS MAPPING REPORT')
print('  (Paper: Valerio et al. 2025 — Section 3.4 methodology)')
print('='*70)
print(f'  Patient         : {PAT_NAME}')
print(f'  Model           : AGSE-VNet 3D UNet (Epoch {int(saved_epoch)+1})')
print(f'  Atlas           : Julich Brain Atlas (siibra-python)')
print(f'  Registration    : ANTsPy SyNQuick  →  MNI152 space')
print(f'  Mapping target  : Tumor Core (NCR label-1 + ET label-3)')
print(f'  TC voxels (MNI) : {TC_TOTAL_VOXELS}  (~{TC_TOTAL_VOXELS * VOX_VOL_MM3:.0f} mm³)')
print()
print(f'  {"#":<3} {"Julich Region":<44} {"% Tumor":>8}  {"% Region":>10}')
print('  ' + '-'*68)
for i, r in enumerate(top5, 1):
    print(f'  {i:<3} {r["Region"]:<44} '
          f'{r["Percentage_of_Tumor"]:>7.2f}%  {r["Percentage_of_Region_Affected"]:>9.2f}%')
print()
print('  Dice Scores (this patient, vs ground truth):')
for k, v in RUN_DICE.items():
    print(f'    {k}: {v:.4f}')
print('='*70)

# ── Copy to Drive ──────────────────────────────────────────────────────────
import shutil
outputs = {
    '/content/atlas_mapping_output.json'       : 'atlas_mapping_output.json',
    '/content/atlas_mapping_visualization.png' : 'atlas_mapping_visualization.png',
    '/content/flair_mni.nii.gz'               : 'flair_mni.nii.gz',
    '/content/tc_mask_mni.nii.gz'             : 'tc_mask_mni.nii.gz',
    '/content/ed_mask_mni.nii.gz'             : 'ed_mask_mni.nii.gz',
    '/content/et_mask_mni.nii.gz'             : 'et_mask_mni.nii.gz',
    '/content/seg_pred_mni.nii.gz'            : 'seg_pred_mni.nii.gz',
}
for src, fname in outputs.items():
    if Path(src).exists():
        shutil.copy(src, str(OUTPUT_DIR / fname))
        print(f'  Drive ← {fname}')

print(f'\nAll outputs → {OUTPUT_DIR}')
print('\n✅  Atlas mapping complete. JSON ready for LLM report generation.')


In [ ]:
# Cell 16 — Install BioMistral Dependencies
!pip install -q transformers bitsandbytes accelerate
print('✅ Done')

In [ ]:
# Cell — Resolve Julich Region IDs to Real Anatomical Names
# Run this BEFORE Cell 18 (prompt building)

import siibra
import numpy as np

print('Resolving Julich region IDs to anatomical names...')

# Method 1: Use the parcellation map's label index directly
label_to_anatomical = {}

try:
    # Fetch the labelled volume index from siibra
    # pmap was already fetched in Cell 10
    vol = pmap.fetch()

    # siibra stores a label-to-region index in the image header or via decode
    if hasattr(pmap, 'decode'):
        for atlas_id in np.unique(atlas_data[atlas_data > 0]):
            try:
                region = pmap.decode(int(atlas_id))
                if region:
                    label_to_anatomical[int(atlas_id)] = region.name
            except:
                pass
    print(f'  Method 1 resolved: {len(label_to_anatomical)} names')
except Exception as e:
    print(f'  Method 1 failed: {e}')

# Method 2: Walk all regions in the parcellation and match by map index
if len(label_to_anatomical) < 5:
    try:
        for region in parcellation:
            try:
                indices = region.get_regional_map(space_mni, maptype='labelled')
                if indices:
                    for idx in indices:
                        label_to_anatomical[int(idx)] = region.name
            except:
                # Fallback: match by region name to known Julich IDs
                for atlas_id in np.unique(atlas_data[atlas_data > 0]):
                    try:
                        r = parcellation.get_region(atlas_id)
                        if r:
                            label_to_anatomical[int(atlas_id)] = r.name
                    except:
                        pass
                break
        print(f'  Method 2 resolved: {len(label_to_anatomical)} names')
    except Exception as e:
        print(f'  Method 2 failed: {e}')

# Method 3: Most reliable — use siibra's assign() function directly on TC voxel coords
if len(label_to_anatomical) < 5:
    try:
        print('  Trying siibra.assign() on TC centroid coordinates...')
        target_ids = [98, 68, 124, 158, 186]   # ← your top-5 IDs

        for atlas_id in target_ids:
            # Find a voxel with this atlas label and convert to MNI mm coords
            vox_matches = np.argwhere(atlas_data == atlas_id)
            if len(vox_matches) == 0:
                continue
            # Take centroid voxel
            centroid_vox = vox_matches[len(vox_matches)//2]
            # Convert voxel → MNI mm using atlas affine
            centroid_mm  = (atlas_affine @ np.append(centroid_vox, 1))[:3]

            # Query siibra for region at this MNI coordinate
            pt = siibra.Point(centroid_mm.tolist(), space=space_mni)
            assignments = siibra.assign(pt, parcellation)
            if assignments:
                best = assignments.iloc[0]
                region_name = best['region'].name if hasattr(best['region'], 'name') else str(best['region'])
                label_to_anatomical[atlas_id] = region_name
                print(f'    ID {atlas_id} → {region_name}')

        print(f'  Method 3 resolved: {len(label_to_anatomical)} names')
    except Exception as e:
        print(f'  Method 3 failed: {e}')

# Method 4: Hard fallback — use siibra's known Julich v3.1 label list
if len(label_to_anatomical) < 5:
    print('  Using siibra region iterator as final fallback...')
    try:
        all_regions = list(parcellation)
        print(f'  Total regions in parcellation: {len(all_regions)}')

        # Build a flat name list ordered by index
        for i, region in enumerate(all_regions):
            label_to_anatomical[i + 1] = region.name

        print(f'  Fallback mapped {len(label_to_anatomical)} regions by order')
    except Exception as e:
        print(f'  Method 4 failed: {e}')

# ── Patch the top5 list and the JSON ──────────────────────────────────────
print('\nFinal resolved names:')
for entry in atlas_json['MRI_Scan']['Tumor_Details']['Spatial_Distribution']:
    raw = entry['Region']                           # e.g. "Julich_Region_98"
    atlas_id = int(raw.split('_')[-1])

    resolved = label_to_anatomical.get(atlas_id)
    if resolved:
        entry['Region'] = resolved
        print(f'  {atlas_id:>4} → {resolved}')
    else:
        print(f'  {atlas_id:>4} → (unresolved — keeping {raw})')

# Also patch top5 list used in visualization
for r in top5:
    atlas_id = r['atlas_id']
    if atlas_id in label_to_anatomical:
        r['Region'] = label_to_anatomical[atlas_id]

# Re-save the updated JSON
with open('/content/atlas_mapping_output.json', 'w') as f:
    json.dump(atlas_json, f, indent=2)
print('\nJSON updated with anatomical names.')

In [ ]:
# Cell 19 — Build Dual-Audience Medical Report Prompt
import json
with open('/content/atlas_mapping_output.json') as f:
    atlas_json = json.load(f)

json_str = json.dumps(atlas_json, indent=2)

# Enhanced prompt for dual-audience reporting
PROMPT = f"""Using the provided JSON data from a brain MRI study, generate a comprehensive medical report that serves BOTH medical professionals and patients/families. Structure the report as follows:

**SECTION 1: EXECUTIVE SUMMARY (For Everyone)**
Write a 3-4 paragraph plain-language overview that anyone can understand:
- What was found in the scan (avoiding medical jargon)
- Where the tumor is located (use everyday language like "left side of the brain near the area that controls movement")
- What this might mean for the patient
- Use analogies when helpful (e.g., "the size of a walnut", "affecting an area about the size of a quarter")

**SECTION 2: CLINICAL FINDINGS (For Medical Professionals)**
Provide detailed clinical analysis including:
- Precise anatomical localization using Julich Brain Atlas nomenclature
- Quantitative tumor burden (total voxel count, percentage of regional involvement)
- Semantic segmentation breakdown (tumor core, peritumoral edema, GD-enhancing tumor) with clinical significance
- Model confidence and segmentation metrics
- Differential considerations based on spatial distribution

**SECTION 3: REGIONAL IMPACT ANALYSIS (Dual-Audience)**
For each affected brain region:

**For Clinicians:**
- Region name (Julich Atlas terminology)
- Percentage of tumor within region
- Percentage of region affected
- Known functional connectivity and networks involved

**For Patients/Families:**
- What this brain area normally does (in simple terms)
- How the tumor might affect these functions
- What symptoms might be related to this area

**SECTION 4: WHAT THIS MEANS (Patient-Focused)**
Explain in everyday language:
- Potential symptoms related to tumor location
- Why certain brain regions are more concerning than others
- What "edema" (swelling) means and why it matters
- Why certain treatments might be recommended based on location

**SECTION 5: NEXT STEPS & RECOMMENDATIONS (For Everyone)**
- Further diagnostic tests that may be needed (and why, in simple terms)
- Treatment considerations based on tumor location
- Questions patients should ask their medical team
- Resources for understanding their condition

**WRITING GUIDELINES:**
- Use SHORT sentences in patient sections
- Define all medical terms the first time they appear (e.g., "edema (swelling around the tumor)")
- Include word count targets: Executive Summary (200-300 words), Clinical Findings (300-400 words), Regional Impact (400-500 words)
- Use empathetic, reassuring tone in patient sections while remaining factually accurate
- In clinical sections, maintain professional medical language and precision
- Include specific percentages and measurements for clinical validation
- Add practical context (e.g., "This region is often involved in language - you might notice difficulty finding words")

JSON DATA:
{json_str}

REPORT:"""

print(f'Prompt length : {len(PROMPT.split())} words')
print(f'Top-5 regions : {[r["Region"] for r in atlas_json["MRI_Scan"]["Tumor_Details"]["Spatial_Distribution"]]}')
print('\n✓ Dual-audience prompt ready')

In [ ]:
# Setup Drive Cache for BioMistral
import os

cache_dir = "/content/drive/MyDrive/huggingface_cache/biomistral"
os.makedirs(cache_dir, exist_ok=True)
os.environ['HF_HOME'] = cache_dir
os.environ['TRANSFORMERS_CACHE'] = cache_dir

print(f"✓ Cache directory set: {cache_dir}")

In [ ]:
# Cell 20 — Load BioMistral with Smart Cache (Drive + Local)
import glob
import shutil
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

print('Loading BioMistral/BioMistral-7B-SLERP in 4-bit...\n')

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "BioMistral/BioMistral-7B-SLERP"

# Define cache paths
local_cache = "/root/.cache/huggingface"
drive_cache = cache_dir  # Defined in Cell 17

model_hash = "models--BioMistral--BioMistral-7B-SLERP"
drive_model_path = os.path.join(drive_cache, "hub", model_hash)
local_model_path = os.path.join(local_cache, "hub", model_hash)

# ─────────────────────────────────────────────────────────────
# Step 1: Check Drive cache and copy to local RAM if available
# ─────────────────────────────────────────────────────────────
if os.path.exists(drive_model_path):
    if not os.path.exists(local_model_path):
        print("📦 Found model in Drive → Copying to local RAM cache...")
        print("   (2-3 min one-time copy, then loads in ~30 sec)\n")
        os.makedirs(os.path.dirname(local_model_path), exist_ok=True)
        shutil.copytree(drive_model_path, local_model_path)
        print("✓ Copy complete! Model ready in local cache.\n")
    else:
        print("✓ Model already in local RAM cache (fastest loading)\n")

    # Load from local cache (30 sec)
    load_cache = local_cache

else:
    # Model not in Drive → will download
    print("⬇️ Model not in Drive cache → Downloading (14.5GB, ~5-10 min)...")
    print("   This happens ONCE. Will auto-backup to Drive after download.\n")
    load_cache = local_cache

# ─────────────────────────────────────────────────────────────
# Step 2: Load tokenizer and model
# ─────────────────────────────────────────────────────────────
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=load_cache
)
print("✓ Tokenizer loaded")

print("\nLoading model (this takes a moment)...")
llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=load_cache,
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

print(f"\n✓ Model loaded successfully!")
print(f"   Device: {llm.device}")
print(f"   Memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

# ─────────────────────────────────────────────────────────────
# Step 3: Backup to Drive if not already there
# ─────────────────────────────────────────────────────────────
if not os.path.exists(drive_model_path) and os.path.exists(local_model_path):
    print("\n💾 Backing up model to Drive for future sessions...")
    os.makedirs(os.path.dirname(drive_model_path), exist_ok=True)
    shutil.copytree(local_model_path, drive_model_path)
    print("✓ Backup complete! Next run will be much faster.")

In [ ]:
# Cell 21 — Generate Medical Report with BioMistral
import time, re
import torch

print('🔬 GENERATING MEDICAL REPORT')
print('=' * 70)
print(f'Model: {model_name} | Device: {llm.device}')
print('=' * 70 + '\n')

inputs = tokenizer(PROMPT, return_tensors='pt').to(llm.device)
input_length = inputs['input_ids'].shape[1]

print(f'Prompt: {input_length} tokens | Generating up to 1024 new tokens...\n')
t0 = time.time()

with torch.no_grad():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    output_ids = llm.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.6,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        early_stopping=True,
    )

generated_ids = output_ids[0][input_length:]
raw_report = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

# ── Clean markdown artifacts from LLM output ──────────────────
def clean_report(text):
    text = re.sub(r'#+\s*', '', text)                        # remove # ## ###
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)          # **bold** → bold
    text = re.sub(r'\*([^*]+)\*', r'\1', text)              # *italic* → italic
    text = re.sub(r'^[-─═]{3,}$', '', text, flags=re.MULTILINE)  # stray divider lines
    text = re.sub(r'\n{3,}', '\n\n', text)                  # max 2 blank lines
    return text.strip()

report = clean_report(raw_report)
# ──────────────────────────────────────────────────────────────

generation_time = time.time() - t0
word_count  = len(report.split())
token_count = len(generated_ids)
MODEL_ID    = model_name

print('✅ COMPLETE!')
print('─' * 70)
print(f'Time: {generation_time:.1f}s | Tokens: {token_count} | Words: {word_count}')
print(f'Speed: {token_count/generation_time:.1f} tokens/sec')
print('─' * 70)

print('\n' + '=' * 70)
print('  📋 MEDICAL REPORT')
print('=' * 70)
print(report)
print('=' * 70)

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# Cell 22 — Save Medical Report
import shutil
from datetime import datetime
from pathlib import Path
import json

print('💾 SAVING REPORTS')
print('=' * 70)

timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
date_readable = datetime.now().strftime('%B %d, %Y at %I:%M %p')

if 'atlas_json' not in globals():
    with open('/content/atlas_mapping_output.json') as f:
        atlas_json = json.load(f)

patient_id = atlas_json['MRI_Scan']['Patient']
model_used = atlas_json['MRI_Scan']['Tumor_Details']['Model_Used']
# Corrected: Use the TC_TOTAL_VOXELS variable calculated earlier
total_voxels = TC_TOTAL_VOXELS

# Save TXT
txt_filename = f'Medical_Report_{patient_id}_{timestamp}.txt'
txt_path = f'/content/{txt_filename}'

with open(txt_path, 'w', encoding='utf-8') as f:
    f.write('═' * 70 + '\n')
    f.write('      AI-ASSISTED MEDICAL IMAGING REPORT\n')
    f.write('═' * 70 + '\n\n')
    f.write(f'Patient ID       : {patient_id}\n')
    f.write(f'Report Generated : {date_readable}\n')
    f.write(f'Segmentation     : {model_used}\n')
    f.write(f'Report Generator : {MODEL_ID}\n')
    f.write(f'Tumor Volume     : {total_voxels:,} voxels\n')
    f.write('─' * 70 + '\n\n')
    f.write('DISCLAIMER: AI-assisted analysis. Requires clinical validation.\n')
    f.write('─' * 70 + '\n\n')
    f.write(report)
    f.write('\n\n' + '═' * 70 + '\n')
    f.write(f'Generated: {date_readable}\n')

shutil.copy(txt_path, str(OUTPUT_DIR / txt_filename))
print(f'✓ TXT: {txt_filename}')

# Save JSON
json_data = {
    'report_id': f'{patient_id}_{timestamp}',
    'patient_id': patient_id,
    'date': date_readable,
    'models': {'segmentation': model_used, 'llm': MODEL_ID},
    'report': report
}

json_filename = f'Report_{patient_id}_{timestamp}.json'
json_path = f'/content/{json_filename}'

with open(json_path, 'w') as f:
    json.dump(json_data, f, indent=2)

shutil.copy(json_path, str(OUTPUT_DIR / json_filename))
print(f'✓ JSON: {json_filename}')

print('=' * 70)
print(f'✅ Saved to: {OUTPUT_DIR}')

In [ ]:
# Cell — Complete Clinical Dashboard (Stats + Report + Images)
import json
import base64
from IPython.display import display, HTML
from pathlib import Path

# ═══════════════════════════════════════════════════════════════
# Load Data
# ═══════════════════════════════════════════════════════════════

# Load atlas mapping JSON
with open('/content/atlas_mapping_output.json') as f:
    atlas_json = json.load(f)

patient_id = atlas_json['MRI_Scan']['Patient']
model_used = atlas_json['MRI_Scan']['Tumor_Details']['Model_Used']
total_voxels = atlas_json['MRI_Scan']['Tumor_Details']['Total_Tumor_Voxels']
regions = atlas_json['MRI_Scan']['Tumor_Details']['Spatial_Distribution']

# Load medical report if available
report_text = ""
if 'report' in globals():
    report_text = report
else:
    # Try to load from file
    report_files = list(Path('/content/drive/MyDrive/AGSE_AtlasMapping_Results').glob('Medical_Report_*.txt'))
    if report_files:
        with open(report_files[0], 'r') as f:
            lines = f.readlines()
            # Skip header, get main content
            start_idx = next((i for i, line in enumerate(lines) if 'DETAILED REPORT' in line), 0)
            report_text = ''.join(lines[start_idx+2:]) if start_idx else ''.join(lines)
    else:
        report_text = "Medical report not yet generated. Run BioMistral cells first."

# Load visualization image (convert to base64)
viz_image_b64 = ""
viz_path = '/content/atlas_mapping_visualization.png'
if Path(viz_path).exists():
    with open(viz_path, 'rb') as f:
        viz_image_b64 = base64.b64encode(f.read()).decode('utf-8')

# Prepare region data
region_data_js = []
for i, r in enumerate(regions[:5], 1):
    region_data_js.append({
        'rank': i,
        'name': r['Region'][:60],
        'tumor_voxels': r['tc_voxels'],
        'pct_tumor': round(float(r['Percentage_of_Tumor'].replace('%', '')), 1),
        'pct_region': round(float(r['Percentage_of_Region_Affected'].replace('%', '')), 1),
    })

region_json = json.dumps(region_data_js)
report_json = json.dumps(report_text)

# Extract and convert percentages for the overview stats (primary region)
primary_region_pct_tumor = round(float(regions[0]['Percentage_of_Tumor'].replace('%', '')), 1)
primary_region_pct_affected = round(float(regions[0]['Percentage_of_Region_Affected'].replace('%', '')), 1)

# ═══════════════════════════════════════════════════════════════
# Create Complete Clinical Dashboard
# ═══════════════════════════════════════════════════════════════

html_dashboard = f"""
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
* {{
    margin: 0;
    padding: 0;
    box-sizing: border-box;
}}

body {{
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Arial, sans-serif;
    background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
    padding: 20px;
    line-height: 1.6;
}}

.dashboard {{
    max-width: 1400px;
    margin: 0 auto;
    background: white;
    border-radius: 16px;
    box-shadow: 0 20px 60px rgba(0,0,0,0.3);
    overflow: hidden;
}}

.header {{
    background: linear-gradient(135deg, #1a202c 0%, #2d3748 100%);
    color: white;
    padding: 40px;
    text-align: center;
    position: relative;
}}

.header::before {{
    content: '';
    position: absolute;
    top: 0;
    left: 0;
    right: 0;
    height: 4px;
    background: linear-gradient(90deg, #667eea, #764ba2, #f093fb);
}}

.header h1 {{
    font-size: 32px;
    margin-bottom: 10px;
    font-weight: 700;
}}

.header .subtitle {{
    font-size: 16px;
    opacity: 0.9;
    margin-top: 8px;
}}

.header .patient-id {{
    display: inline-block;
    background: rgba(102, 126, 234, 0.2);
    padding: 8px 16px;
    border-radius: 20px;
    margin-top: 15px;
    font-size: 14px;
    font-weight: 600;
}}

.tab-nav {{
    display: flex;
    background: #f7fafc;
    border-bottom: 2px solid #e2e8f0;
    overflow-x: auto;
}}

.tab-button {{
    flex: 1;
    padding: 18px 24px;
    background: none;
    border: none;
    cursor: pointer;
    font-size: 15px;
    font-weight: 600;
    color: #718096;
    transition: all 0.3s;
    position: relative;
    min-width: 150px;
}}

.tab-button:hover {{
    background: rgba(102, 126, 234, 0.05);
    color: #667eea;
}}

.tab-button.active {{
    color: #667eea;
    background: white;
}}

.tab-button.active::after {{
    content: '';
    position: absolute;
    bottom: -2px;
    left: 0;
    right: 0;
    height: 3px;
    background: linear-gradient(90deg, #667eea, #764ba2);
}}

.tab-icon {{
    margin-right: 8px;
    font-size: 18px;
}}

.tab-content {{
    display: none;
    animation: fadeIn 0.4s ease-in;
}}

.tab-content.active {{
    display: block;
}}

@keyframes fadeIn {{
    from {{ opacity: 0; transform: translateY(10px); }}
    to {{ opacity: 1; transform: translateY(0); }}
}}

.stats-grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
    gap: 25px;
    padding: 40px;
    background: #f7fafc;
}}

.stat-card {{
    background: white;
    padding: 30px;
    border-radius: 12px;
    box-shadow: 0 4px 12px rgba(0,0,0,0.08);
    border-left: 4px solid #667eea;
    transition: all 0.3s;
}}

.stat-card:hover {{
    transform: translateY(-5px);
    box-shadow: 0 12px 28px rgba(0,0,0,0.15);
}}

.stat-icon {{
    font-size: 36px;
    margin-bottom: 15px;
}}

.stat-label {{
    font-size: 13px;
    text-transform: uppercase;
    color: #718096;
    font-weight: 600;
    margin-bottom: 10px;
    letter-spacing: 0.5px;
}}

.stat-value {{
    font-size: 36px;
    font-weight: 700;
    color: #2d3748;
    margin-bottom: 5px;
}}

.stat-unit {{
    font-size: 14px;
    color: #a0aec0;
}}

.regions-container {{
    padding: 40px;
}}

.section-title {{
    font-size: 24px;
    font-weight: 700;
    color: #2d3748;
    margin-bottom: 30px;
    padding-bottom: 15px;
    border-bottom: 3px solid #667eea;
    display: flex;
    align-items: center;
}}

.section-title::before {{
    content: '📍';
    margin-right: 12px;
    font-size: 28px;
}}

.region-card {{
    background: white;
    padding: 25px;
    margin-bottom: 20px;
    border-radius: 12px;
    border: 2px solid #e2e8f0;
    transition: all 0.3s;
    cursor: pointer;
}}

.region-card:hover {{
    border-color: #667eea;
    box-shadow: 0 8px 24px rgba(102, 126, 234, 0.15);
    transform: translateX(8px);
}}

.region-header {{
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 20px;
}}

.region-rank {{
    width: 50px;
    height: 50px;
    background: linear-gradient(135deg, #667eea, #764ba2);
    color: white;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 700;
    font-size: 22px;
    box-shadow: 0 4px 12px rgba(102, 126, 234, 0.4);
}}

.region-name {{
    flex: 1;
    margin-left: 20px;
    font-size: 17px;
    font-weight: 600;
    color: #2d3748;
}}

.region-voxels {{
    font-size: 15px;
    color: #718096;
    font-weight: 600;
    background: #f7fafc;
    padding: 6px 12px;
    border-radius: 6px;
}}

.region-bars {{
    display: grid;
    grid-template-columns: 1fr 1fr;
    gap: 20px;
    margin-top: 20px;
}}

.bar-container {{
    background: #f7fafc;
    padding: 15px;
    border-radius: 10px;
}}

.bar-label {{
    font-size: 11px;
    color: #718096;
    text-transform: uppercase;
    margin-bottom: 10px;
    font-weight: 600;
    letter-spacing: 0.5px;
}}

.bar-wrapper {{
    background: #e2e8f0;
    height: 28px;
    border-radius: 14px;
    overflow: hidden;
    position: relative;
}}

.bar-fill {{
    height: 100%;
    background: linear-gradient(90deg, #667eea, #764ba2);
    border-radius: 14px;
    transition: width 1.2s cubic-bezier(0.4, 0, 0.2, 1);
    display: flex;
    align-items: center;
    justify-content: flex-end;
    padding-right: 12px;
}}

.bar-fill.alt {{
    background: linear-gradient(90deg, #48bb78, #38a169);
}}

.bar-text {{
    font-size: 13px;
    font-weight: 700;
    color: white;
    text-shadow: 0 1px 2px rgba(0,0,0,0.3);
}}

.viz-container {{
    padding: 40px;
    text-align: center;
}}

.viz-image {{
    max-width: 100%;
    height: auto;
    border-radius: 12px;
    box-shadow: 0 8px 24px rgba(0,0,0,0.15);
    margin-top: 20px;
}}

.viz-caption {{
    margin-top: 20px;
    font-size: 14px;
    color: #718096;
    font-style: italic;
}}

.report-container {{
    padding: 40px;
    max-width: 900px;
    margin: 0 auto;
}}

.report-content {{
    background: white;
    padding: 40px;
    border-radius: 12px;
    border: 1px solid #e2e8f0;
    line-height: 1.8;
    color: #2d3748;
    font-size: 15px;
}}

.report-content h2, .report-content h3 {{
    color: #667eea;
    margin-top: 30px;
    margin-bottom: 15px;
    font-weight: 700;
}}

.report-content h2 {{
    font-size: 22px;
    border-bottom: 2px solid #e2e8f0;
    padding-bottom: 10px;
}}

.report-content h3 {{
    font-size: 18px;
}}

.report-content p {{
    margin-bottom: 15px;
    text-align: justify;
}}

.report-content strong {{
    color: #2d3748;
    font-weight: 600;
}}

.report-disclaimer {{
    background: #fff3cd;
    border-left: 4px solid #ffc107;
    padding: 20px;
    margin: 30px 0;
    border-radius: 8px;
}}

.report-disclaimer strong {{
    color: #856404;
    display: block;
    margin-bottom: 10px;
}}

.footer {{
    background: #f7fafc;
    padding: 25px;
    text-align: center;
    color: #718096;
    font-size: 13px;
    border-top: 1px solid #e2e8f0;
}}

.footer-links {{
    margin-top: 10px;
}}

.footer-links a {{
    color: #667eea;
    text-decoration: none;
    margin: 0 10px;
    font-weight: 600;
}}

.footer-links a:hover {{
    text-decoration: underline;
}}

@media print {{
    body {{
        background: white;
        padding: 0;
    }}
    .tab-nav {{
        display: none;
    }}
    .tab-content {{
        display: block !important;
    }}
}}

.region-card {{
    animation: slideIn 0.5s ease-out;
    opacity: 0;
    animation-fill-mode: forwards;
}}

.region-card:nth-child(1) {{ animation-delay: 0.1s; }}
.region-card:nth-child(2) {{ animation-delay: 0.2s; }}
.region-card:nth-child(3) {{ animation-delay: 0.3s; }}
.region-card:nth-child(4) {{ animation-delay: 0.4s; }}
.region-card:nth-child(5) {{ animation-delay: 0.5s; }}

@keyframes slideIn {{
    from {{
        opacity: 0;
        transform: translateY(30px);
    }}
    to {{
        opacity: 1;
        transform: translateY(0);
    }}
}}
</style>
</head>
<body>
<div class="dashboard">
    <div class="header">
        <h1>🧠 Clinical Brain Tumor Analysis Dashboard</h1>
        <div class="subtitle">AI-Assisted Tumor Segmentation & Atlas Mapping</div>
        <div class="patient-id">Patient: {patient_id} | Model: {model_used}</div>
    </div>

    <div class="tab-nav">
        <button class="tab-button active" onclick="switchTab(0)">
            <span class="tab-icon">📊</span> Overview
        </button>
        <button class="tab-button" onclick="switchTab(1)">
            <span class="tab-icon">🗺️</span> Regional Analysis
        </button>
        <button class="tab-button" onclick="switchTab(2)">
            <span class="tab-icon">🖼️</span> Visualization
        </button>
        <button class="tab-button" onclick="switchTab(3)">
            <span class="tab-icon">📄</span> Medical Report
        </button>
    </div>

    <div class="tab-content active" id="tab-0">
        <div class="stats-grid">
            <div class="stat-card">
                <div class="stat-icon">🎯</div>
                <div class="stat-label">Total Tumor Volume</div>
                <div class="stat-value">{total_voxels:,}</div>
                <div class="stat-unit">voxels (MNI space)</div>
            </div>

            <div class="stat-card" style="border-left-color: #48bb78;">
                <div class="stat-icon">📍</div>
                <div class="stat-label">Regions Affected</div>
                <div class="stat-value">{len(regions)}</div>
                <div class="stat-unit">Julich Atlas regions</div>
            </div>

            <div class="stat-card" style="border-left-color: #ed8936;">
                <div class="stat-icon">⭐</div>
                <div class="stat-label">Primary Region</div>
                <div class="stat-value">{primary_region_pct_tumor:.1f}%</div>
                <div class="stat-unit">of total tumor mass</div>
            </div>

            <div class="stat-card" style="border-left-color: #9f7aea;">
                <div class="stat-icon">🧬</div>
                <div class="stat-label">Segmentation Model</div>
                <div class="stat-value" style="font-size: 18px;">{model_used}</div>
                <div class="stat-unit">3D attention-gated encoder</div>
            </div>
        </div>

        <div class="regions-container">
            <h2 class="section-title">Summary: Top Affected Region</h2>
            <div class="region-card" style="border-color: #667eea;">
                <div class="region-header">
                    <div class="region-rank">1</div>
                    <div class="region-name">{regions[0]['Region']}</div>
                    <div class="region-voxels">{regions[0]['tc_voxels']:,} voxels</div>
                </div>
                <div style="font-size: 14px; color: #718096; line-height: 1.6;">
                    This region contains <strong>{primary_region_pct_tumor:.1f}%</strong> of the total tumor volume,
                    with <strong>{primary_region_pct_affected:.1f}%</strong> of the region's total tissue affected.
                </div>
            </div>
        </div>
    </div>

    <div class="tab-content" id="tab-1">
        <div class="regions-container">
            <h2 class="section-title">Top 5 Brain Regions by Tumor Burden</h2>
            <div id="regions-list"></div>
        </div>
    </div>

    <div class="tab-content" id="tab-2">
        <div class="viz-container">
            <h2 class="section-title">MRI Atlas Overlay Visualization</h2>
            {"<img src='data:image/png;base64," + viz_image_b64 + "' class='viz-image' alt='Atlas Mapping Visualization'>" if viz_image_b64 else "<p style='color: #718096; padding: 60px;'>Visualization not available. Run Cell 14 first.</p>"}
            <p class="viz-caption">
                3-plane view: Tumor core (red) overlaid on Julich Brain Atlas regions (MNI152 space)<br>
                Axial, Coronal, and Sagittal slices at tumor centroid
            </p>
        </div>
    </div>

    <div class="tab-content" id="tab-3">
        <div class="report-container">
            <h2 class="section-title">AI-Generated Medical Report</h2>
            <div class="report-disclaimer">
                <strong>⚠️ Clinical Disclaimer</strong>
                This report is generated using AI-assisted analysis tools and is intended to SUPPORT,
                not replace, clinical judgment. All findings must be reviewed and validated by qualified
                medical professionals.
            </div>
            <div class="report-content" id="report-content"></div>
        </div>
    </div>

    <div class="footer">
        <strong>Technology Stack:</strong> AGSE-VNet 3D U-Net | Julich Brain Cytoarchitectonic Atlas v3.1 |
        ANTsPy Registration | MNI152 2009c Nonlinear Asymmetric Space | BioMistral-7B Medical LLM
        <div class="footer-links">
            <a href="#" onclick="window.print(); return false;">🖨️ Print Dashboard</a>
            <a href="#" onclick="downloadReport(); return false;">💾 Download Report</a>
        </div>
    </div>
</div>

<script>
const regions = {region_json};
const reportText = {report_json};

function switchTab(index) {{
    document.querySelectorAll('.tab-content').forEach(tab => {{ tab.classList.remove('active'); }});
    document.querySelectorAll('.tab-button').forEach(btn => {{ btn.classList.remove('active'); }});
    document.getElementById('tab-' + index).classList.add('active');
    document.querySelectorAll('.tab-button')[index].classList.add('active');
    if (index === 1) {{ setTimeout(animateBars, 100); }}
}}

const container = document.getElementById('regions-list');
regions.forEach((region, index) => {{
    const card = document.createElement('div');
    card.className = 'region-card';
    card.innerHTML = `
        <div class="region-header">
            <div class="region-rank">${{region.rank}}</div>
            <div class="region-name">${{region.name}}</div>
            <div class="region-voxels">${{region.tumor_voxels.toLocaleString()}} voxels</div>
        </div>
        <div class="region-bars">
            <div class="bar-container">
                <div class="bar-label">Percentage of Total Tumor</div>
                <div class="bar-wrapper">
                    <div class="bar-fill" style="width: 0%" data-width="${{region.pct_tumor}}%">
                        <span class="bar-text">${{region.pct_tumor}}%</span>
                    </div>
                </div>
            </div>
            <div class="bar-container">
                <div class="bar-label">Percentage of Region Affected</div>
                <div class="bar-wrapper">
                    <div class="bar-fill alt" style="width: 0%" data-width="${{region.pct_region}}%">
                        <span class="bar-text">${{region.pct_region}}%</span>
                    </div>
                </div>
            </div>
        </div>
    `;
    container.appendChild(card);
}});

function animateBars() {{
    document.querySelectorAll('.bar-fill').forEach(bar => {{
        bar.style.width = bar.dataset.width;
    }});
}}
setTimeout(animateBars, 100);

function formatReport(text) {{
    let html = text
        .replace(/SECTION (\\d+):/g, '<h2>SECTION $1:</h2>')
        .replace(/\\*\\*([^*]+)\\*\\*/g, '<strong>$1</strong>')
        .replace(/\\n\\n/g, '</p><p>')
        .replace(/^([^<])/gm, '<p>$1');
    return '<div>' + html + '</div>';
}}
document.getElementById('report-content').innerHTML = formatReport(reportText);

function downloadReport() {{
    const reportContent = `\nBRAIN TUMOR ANALYSIS REPORT\n═══════════════════════════════════════════════════════════════\n\nPatient: {patient_id}\nModel: {model_used}\nTotal Tumor Volume: {total_voxels:,} voxels\n\n${{reportText}}\n\n═══════════════════════════════════════════════════════════════\nGenerated using AGSE-VNet + Julich Brain Atlas + BioMistral LLM\n`;
    const blob = new Blob([reportContent], {{ type: 'text/plain' }});
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = 'Clinical_Report_{patient_id}.txt';
    a.click();
    URL.revokeObjectURL(url);
}}
</script>
</body>
</html>
"""

# Display the complete dashboard
display(HTML(html_dashboard))
